# 🏋️ 02 — Entrenamiento del Modelo (MobileNetV2 + Transfer Learning)
**Proyecto 01: Reciclaje Inteligente — Clasificación de Desechos con Visión Artificial**  
Grupo #4 · Curso 045 — Inteligencia Artificial · UMG 2026  
Responsable: Marvin Vásquez / Javier Rivera

---
## Objetivos de este notebook
1. Cargar el dataset procesado (`data/processed/`) con `ImageFolder`
2. Visualizar el efecto del data augmentation en muestras reales
3. **FASE 1** — Entrenar solo el clasificador con la base congelada (5 épocas, lr=1e-3)
4. **FASE 2** — Fine-tuning de las últimas 20 capas (10 épocas, lr=1e-4)
5. Graficar curvas de aprendizaje (loss + accuracy) y guardar el mejor modelo

---
> ⚠️ **Pre-requisito:** Ejecutar `python src/split_dataset.py` antes de correr este notebook.

## 0. Configuración e importaciones

In [ ]:
# ── Instalar dependencias (descomentar si es necesario) ────────────────────
# !pip install torch torchvision matplotlib seaborn pillow numpy scikit-learn

import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import f1_score

# ── Agregar src/ al path ───────────────────────────────────────────────────
sys.path.insert(0, os.path.join('..', 'src'))
from model import build_model, unfreeze_last_layers, CLASSES, COLOR_MAP
from preprocess import get_train_transforms, get_val_transforms

# ── Estilo de gráficas ─────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Rutas ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/processed')
MODEL_PATH = Path('../models/modelo_reciclaje.pth')
DOCS_DIR   = Path('../docs')
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dispositivo ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('✅ Librerías importadas correctamente')
print(f'  Dispositivo: {device}  ({"GPU — entrenamiento rápido 🚀" if device.type == "cuda" else "CPU — puede ser lento ⏳"})')
print(f'  PyTorch: {torch.__version__}')
print(f'  Dataset: {DATA_DIR.resolve()}')
print(f'  Clases del proyecto ({len(CLASSES)}): {CLASSES}')

## 1. Carga del dataset

In [ ]:
# ── Verificar estructura del dataset ──────────────────────────────────────
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        print(f'❌ No se encontró: {split_dir}')
        print('   Ejecuta primero: python src/split_dataset.py')
    else:
        clases = [d.name for d in split_dir.iterdir() if d.is_dir()]
        n_imgs = sum(len(list(d.glob('*.*'))) for d in split_dir.iterdir() if d.is_dir())
        print(f'✅ {split:<6} — {len(clases)} clases — {n_imgs} imágenes')

# ── Cargar datasets con ImageFolder ───────────────────────────────────────
train_ds = ImageFolder(str(DATA_DIR / 'train'), transform=get_train_transforms())
val_ds   = ImageFolder(str(DATA_DIR / 'val'),   transform=get_val_transforms())
test_ds  = ImageFolder(str(DATA_DIR / 'test'),  transform=get_val_transforms())

BATCH_SIZE   = 32
NUM_WORKERS  = 0  # 0 para Windows (evita problemas de multiprocessing)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

print(f'\n📦 Dataset cargado:')
print(f'   Train:      {len(train_ds):>5} imágenes — {len(train_loader)} batches de {BATCH_SIZE}')
print(f'   Validación: {len(val_ds):>5} imágenes — {len(val_loader)} batches')
print(f'   Test:       {len(test_ds):>5} imágenes — {len(test_loader)} batches')
print(f'\n   Clases detectadas por ImageFolder:')
for i, c in enumerate(train_ds.classes):
    n = train_ds.targets.count(i)
    print(f'   [{i}] {c:<28} → {n} imágenes en train')

## 2. Visualización del data augmentation

In [ ]:
# ── Mostrar efecto de augmentation en 4 clases ────────────────────────────
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

def desnormalizar(tensor):
    """Convierte tensor normalizado a imagen PIL para visualización."""
    t = tensor.clone().permute(1, 2, 0)
    t = t * IMAGENET_STD + IMAGENET_MEAN
    t = t.clamp(0, 1)
    return (t.numpy() * 255).astype('uint8')

N_CLASES_MOSTRAR = min(4, len(train_ds.classes))
N_AUGMENTACIONES = 5  # versiones aumentadas de la misma imagen

fig, axes = plt.subplots(N_CLASES_MOSTRAR, N_AUGMENTACIONES + 1,
                         figsize=((N_AUGMENTACIONES + 1) * 2.2, N_CLASES_MOSTRAR * 2.2))

if N_CLASES_MOSTRAR == 1:
    axes = [axes]

for row, clase_idx in enumerate(range(N_CLASES_MOSTRAR)):
    clase_nombre = train_ds.classes[clase_idx]
    # Buscar primera imagen de esta clase
    indices_clase = [i for i, t in enumerate(train_ds.targets) if t == clase_idx]
    if not indices_clase:
        continue

    # Imagen original (sin augmentation)
    ruta_img = train_ds.imgs[indices_clase[0]][0]
    img_orig = Image.open(ruta_img).convert('RGB').resize((224, 224))
    axes[row][0].imshow(img_orig)
    axes[row][0].set_title('Original', fontsize=8, color='gray')
    axes[row][0].axis('off')

    emoji = COLOR_MAP.get(clase_nombre, ('', '', '', ''))[2]
    axes[row][0].set_ylabel(f'{emoji} {clase_nombre}', rotation=0, labelpad=65,
                             va='center', fontsize=8, fontweight='bold')

    # Versiones aumentadas
    aug_transform = get_train_transforms()
    for col in range(1, N_AUGMENTACIONES + 1):
        tensor_aug = aug_transform(img_orig)
        img_aug = desnormalizar(tensor_aug)
        axes[row][col].imshow(img_aug)
        axes[row][col].set_title(f'Aug #{col}', fontsize=8, color='#2E86C1')
        axes[row][col].axis('off')

plt.suptitle('Efecto del Data Augmentation (rotación, flip, brillo, contraste)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(str(DOCS_DIR / 'train_augmentation_preview.png'), bbox_inches='tight')
plt.show()
print('💾 Guardado en docs/train_augmentation_preview.png')

## 3. Construcción del modelo y pérdida ponderada

In [ ]:
# ── Modelo MobileNetV2 con Transfer Learning ───────────────────────────────
model = build_model().to(device)

# Parámetros totales vs entrenable
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'📐 Parámetros totales:      {total_params:>10,}')
print(f'   Parámetros entrenables: {trainable_params:>10,}  ({trainable_params/total_params:.1%} del total)')
print(f'   Base congelada:         {total_params - trainable_params:>10,}')

# ── Pesos por clase (para manejar desbalance) ─────────────────────────────
class_counts = [train_ds.targets.count(i) for i in range(len(train_ds.classes))]
total_imgs   = sum(class_counts)
weights      = torch.tensor(
    [total_imgs / (len(train_ds.classes) * max(c, 1)) for c in class_counts],
    dtype=torch.float
).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

print(f'\n⚖️  Pesos de clase (CrossEntropyLoss):')
for c, w, n in zip(train_ds.classes, weights.cpu().tolist(), class_counts):
    barra = '█' * int(w * 3)
    print(f'   {c:<28} peso={w:.3f}  n={n:>4}  {barra}')

## 4. Funciones de entrenamiento y evaluación

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Realiza una época de entrenamiento y retorna loss y accuracy."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, device):
    """Evalúa el modelo y retorna accuracy y F1-macro."""
    model.eval()
    preds, targets = [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds.extend(outputs.argmax(1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    acc = sum(p == t for p, t in zip(preds, targets)) / len(targets)
    f1  = f1_score(targets, preds, average='macro', zero_division=0)
    return acc, f1, total_loss / len(loader.dataset)


print('✅ Funciones train_epoch() y evaluate() definidas.')

## 5. FASE 1 — Entrenamiento del clasificador (base congelada)

In [ ]:
EPOCHS_FASE1 = 5
LR_FASE1     = 1e-3

optimizer1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FASE1
)
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer1, patience=2, factor=0.5, verbose=True)

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     []
}
best_f1    = 0.0
best_epoch = 0

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"FASE 1 — Solo clasificador ({EPOCHS_FASE1} épocas, lr={LR_FASE1})")
print(f"{'='*60}")
print(f"{'Época':>5} | {'Loss Train':>10} | {'Acc Train':>9} | {'Loss Val':>8} | {'Acc Val':>7} | {'F1-macro':>8} | {'Estado':>8}")
print("-" * 70)

for epoch in range(1, EPOCHS_FASE1 + 1):
    t_start = time.time()
    tr_loss, tr_acc         = train_epoch(model, train_loader, criterion, optimizer1, device)
    val_acc, val_f1, v_loss = evaluate(model, val_loader, device)
    scheduler1.step(v_loss)
    elapsed = time.time() - t_start

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    guardado = ''
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch
        torch.save(model.state_dict(), str(MODEL_PATH))
        guardado = '💾 saved'

    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.3f} | {v_loss:>8.4f} | {val_acc:>7.3f} | {val_f1:>8.3f} | {guardado:>8}  ({elapsed:.0f}s)")

print(f"\n✅ FASE 1 completada — Mejor F1-macro: {best_f1:.3f} (época {best_epoch})")

## 6. FASE 2 — Fine-tuning (últimas 20 capas)

In [ ]:
EPOCHS_FASE2   = 10
LR_FASE2       = 1e-4
N_CAPAS_UNFREEZE = 20

# Descongelar las últimas N capas
model = unfreeze_last_layers(model, n=N_CAPAS_UNFREEZE)
trainable_f2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🔓 Capas descongeladas: últimas {N_CAPAS_UNFREEZE}')
print(f'   Parámetros entrenables ahora: {trainable_f2:,}')

optimizer2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FASE2
)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, patience=3, factor=0.5, verbose=True)

print(f"\n{'='*60}")
print(f"FASE 2 — Fine-tuning ({EPOCHS_FASE2} épocas, lr={LR_FASE2})")
print(f"{'='*60}")
print(f"{'Época':>5} | {'Loss Train':>10} | {'Acc Train':>9} | {'Loss Val':>8} | {'Acc Val':>7} | {'F1-macro':>8} | {'Estado':>8}")
print("-" * 70)

for epoch in range(1, EPOCHS_FASE2 + 1):
    t_start = time.time()
    tr_loss, tr_acc         = train_epoch(model, train_loader, criterion, optimizer2, device)
    val_acc, val_f1, v_loss = evaluate(model, val_loader, device)
    scheduler2.step(v_loss)
    elapsed = time.time() - t_start

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    guardado = ''
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch + EPOCHS_FASE1
        torch.save(model.state_dict(), str(MODEL_PATH))
        guardado = '💾 saved'

    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.3f} | {v_loss:>8.4f} | {val_acc:>7.3f} | {val_f1:>8.3f} | {guardado:>8}  ({elapsed:.0f}s)")

print(f"\n✅ FASE 2 completada")
print(f"   🏆 Mejor F1-macro global: {best_f1:.3f} (época {best_epoch} de {EPOCHS_FASE1 + EPOCHS_FASE2})")
print(f"   💾 Modelo guardado en: {MODEL_PATH.resolve()}")

## 7. Curvas de aprendizaje

In [ ]:
n_total  = len(history['train_loss'])
epocas   = list(range(1, n_total + 1))
sep_fase = EPOCHS_FASE1 + 0.5  # línea divisoria entre fases

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Loss ─────────────────────────────────────────────────────────────────
axes[0].plot(epocas, history['train_loss'], 'o-', color='#E74C3C', label='Train', linewidth=2, markersize=4)
axes[0].plot(epocas, history['val_loss'],   's-', color='#2E86C1', label='Validación', linewidth=2, markersize=4)
axes[0].axvline(sep_fase, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Fase 1 → 2')
axes[0].set_title('Pérdida (Loss) por época', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# ── Accuracy ─────────────────────────────────────────────────────────────
axes[1].plot(epocas, [a * 100 for a in history['train_acc']], 'o-', color='#E74C3C', label='Train', linewidth=2, markersize=4)
axes[1].plot(epocas, [a * 100 for a in history['val_acc']],   's-', color='#2E86C1', label='Validación', linewidth=2, markersize=4)
axes[1].axvline(sep_fase, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Fase 1 → 2')
axes[1].set_title('Accuracy (%) por época', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(True, alpha=0.4)

# ── F1-macro ─────────────────────────────────────────────────────────────
axes[2].plot(epocas, [f * 100 for f in history['val_f1']], 'D-', color='#27AE60', label='F1-macro Val', linewidth=2, markersize=4)
axes[2].axhline(best_f1 * 100, color='#F39C12', linestyle=':', linewidth=1.5, label=f'Mejor F1: {best_f1*100:.1f}%')
axes[2].axvline(sep_fase, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Fase 1 → 2')
axes[2].set_title('F1-macro (%) por época', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('F1-macro (%)')
axes[2].set_ylim(0, 105)
axes[2].legend()
axes[2].grid(True, alpha=0.4)

plt.suptitle(
    f'Curvas de Aprendizaje — MobileNetV2 · Reciclaje Inteligente · Grupo #4 · UMG 2026\n'
    f'Fase 1: {EPOCHS_FASE1} épocas (lr={LR_FASE1}) + Fase 2: {EPOCHS_FASE2} épocas (lr={LR_FASE2})',
    fontsize=11, color='gray', y=1.02
)
plt.tight_layout()

save_path = str(DOCS_DIR / 'train_curvas_aprendizaje.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Curvas guardadas en: {save_path}')

## 8. Resumen final del entrenamiento

In [ ]:
print('=' * 60)
print('  RESUMEN DE ENTRENAMIENTO — Grupo #4 · UMG 2026')
print('=' * 60)
print(f'\n  Modelo:          MobileNetV2 (preentrenado en ImageNet)')
print(f'  Clases:          {len(train_ds.classes)}')
print(f'  Imágenes train:  {len(train_ds)}')
print(f'  Imágenes val:    {len(val_ds)}')
print(f'  Épocas totales:  {EPOCHS_FASE1 + EPOCHS_FASE2} (Fase 1: {EPOCHS_FASE1} + Fase 2: {EPOCHS_FASE2})')
print(f'\n  🏆 Mejor F1-macro en validación: {best_f1*100:.2f}%  (época {best_epoch})')
print(f'  🎯 Accuracy final en validación: {history["val_acc"][-1]*100:.2f}%')

# Diagnóstico de overfitting
diff_acc = history['train_acc'][-1] - history['val_acc'][-1]
if diff_acc > 0.15:
    print(f'\n  ⚠️  POSIBLE OVERFITTING detectado')
    print(f'     Diferencia train-val acc: {diff_acc:.1%}')
    print(f'     → Considera: más augmentation, mayor dropout, o más datos')
else:
    print(f'\n  ✅ Sin signos de overfitting severo (diff acc: {diff_acc:.1%})')

print(f'\n  📁 Archivos generados:')
archivos = [
    ('models/modelo_reciclaje.pth', MODEL_PATH),
    ('docs/train_augmentation_preview.png', DOCS_DIR / 'train_augmentation_preview.png'),
    ('docs/train_curvas_aprendizaje.png',   DOCS_DIR / 'train_curvas_aprendizaje.png'),
]
for nombre, ruta in archivos:
    existe = '✅' if ruta.exists() else '❌'
    print(f'     {existe} {nombre}')

print(f'\n  ✅ Siguiente paso: notebooks/03_evaluation.ipynb')